# 归一化层详解：BatchNorm / LayerNorm / InstanceNorm
## 核心统一作用
1. 标准化特征分布，缓解深层网络梯度消失/爆炸
2. 允许使用更大学习率，大幅加速模型收敛
3. 弱化权重初始化敏感程度，提升泛化能力
4. 统一流程：标准化 + 可学习缩放$\gamma$ + 可学习偏移$\beta$（InstanceNorm无$\beta$）

通用标准化公式：
$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$
$\epsilon$ 极小值防止分母为0；最终输出 $y=\gamma\hat{x}+\beta$

In [ ]:
# ====================== 全局依赖导入 ======================
import torch
import torch.nn as nn

# 自动设备适配
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"运行设备：{device}")

## 一、nn.BatchNorm1d / nn.BatchNorm2d 批量归一化
### 1. 计算维度规则
- BatchNorm1d：输入形状 `[B, C]` 全连接 / `[B, C, L]` 一维时序卷积
- BatchNorm2d：输入形状 `[B, C, H, W]` 二维图像卷积
**统计维度：沿Batch批量维度，对单个通道C内所有样本做归一化**

以图像 [B,C,H,W] 举例：
1. 分组：按通道C拆分，每个通道独立计算
2. 统计：取当前batch全部B张图片，该通道下所有H*W像素，计算均值$\mu_B$、方差$\sigma_B^2$
3. 标准化：每个像素减去同通道批量均值，除以批量标准差
4. 仿射变换：可训练参数$\gamma$缩放、$\beta$偏移，恢复特征表达能力

### 2. Train / Eval 两种模式行为差异
1. model.train() 训练模式
   - 使用当前batch实时均值、方差做标准化
   - 滑动平均更新全局均值/方差缓存（running_mean / running_var）
2. model.eval() 推理模式
   - 不再统计新batch分布，直接使用训练缓存的全局均值方差
   - 保证单张图片推理结果稳定，不受batch样本影响

### 3. 优缺点 & 适用场景
✅ 优点：大batch下稳定收敛，CNN图像分类标配
❌ 缺点：batch过小时统计分布失真，效果暴跌；单样本推理分布偏移
适用：CNN、全连接分类网络、batch尺寸充足场景

In [ ]:
# ====================== BatchNorm2d 图像卷积示例 ======================
# 参数num_features：通道数量C
bn2d = nn.BatchNorm2d(num_features=3).to(device)
# 模拟输入：batch=4，3通道RGB，32×32图片 [B,C,H,W]
img_batch = torch.randn(4, 3, 32, 32).to(device)

# 1. 训练模式（默认train）
bn2d.train()
out_train = bn2d(img_batch)
print("【Train模式】BatchNorm2d输出shape:", out_train.shape)
print("训练阶段缓存均值:", bn2d.running_mean[:3])

# 2. 推理模式
bn2d.eval()
# 单张图片推理，无batch统计依赖
single_img = torch.randn(1, 3, 32, 32).to(device)
out_eval = bn2d(single_img)
print("【Eval模式】单图输出shape:", out_eval.shape)

# ====================== BatchNorm1d 全连接示例 ======================
bn1d = nn.BatchNorm1d(num_features=16).to(device)
# batch=8，16维特征 [B,C]
fc_input = torch.randn(8, 16).to(device)
out_fc = bn1d(fc_input)
print("BatchNorm1d全连接输出shape:", out_fc.shape)

## 二、nn.LayerNorm 层归一化（Transformer/NLP主流）
### 1. 计算维度规则
输入形状：`[B, seq_len, d_model]` NLP时序；`[B, C]` 全连接
**统计维度：单条样本内部所有特征维度，完全不依赖Batch**

以文本 [B, seq_len, d_model] 举例：
1. 每条样本独立处理，不跨batch交互
2. 对单个样本内全部d_model维度特征，一次性计算均值$\mu_L$、方差$\sigma_L^2$
3. 标准化当前样本所有特征
4. 可学习$\gamma$、$\beta$缩放偏移

### 2. Train / Eval 行为差异
无全局滑动均值缓存！训练、推理计算逻辑完全一致
每条样本单独标准化，单样本推理不受batch大小干扰

### 3. 优缺点 & 适用场景
✅ 优点：不受batch尺寸限制，小batch、单样本推理稳定；适配变长序列
❌ 缺点：图像任务效果弱于BatchNorm，对空间分布建模差
适用：Transformer、BERT、GPT、RNN时序模型、小batch训练场景

In [ ]:
# ====================== LayerNorm Transformer时序示例 ======================
# normalized_shape：需要归一化的特征维度d_model
ln = nn.LayerNorm(normalized_shape=64).to(device)
# 模拟文本输入：batch=2，序列长度10，隐藏维度64 [B, seq_len, d_model]
text_feat = torch.randn(2, 10, 64).to(device)

# 训练/eval逻辑完全相同，无需切换模式改变计算
ln.train()
out_ln_train = ln(text_feat)
print("LayerNorm 训练输出shape:", out_ln_train.shape)

ln.eval()
# 单条文本推理，无batch依赖
single_text = torch.randn(1, 10, 64).to(device)
out_ln_eval = ln(single_text)
print("LayerNorm 单样本推理shape:", out_ln_eval.shape)

# 全连接场景使用LayerNorm
ln_fc = nn.LayerNorm(20)
fc_x = torch.randn(4, 20)
print("LayerNorm全连接输出shape:", ln_fc(fc_x).shape)

## 三、nn.InstanceNorm2d 实例归一化（图像风格迁移专用）
### 1. 计算维度规则
输入固定图像形状 `[B, C, H, W]`，无Instance1d常用场景
**统计维度：单张图片、单个通道内部空间像素(H,W)，彻底剥离Batch信息**

计算流程：
1. 拆分到最小单元：每张图片 + 单个通道独立计算
2. 仅取该图该通道内H×W所有像素计算均值方差
3. 标准化该通道像素；仅带可学习缩放$\gamma$，**无偏移参数$\beta$**

### 2. Train / Eval 行为差异
无全局滑动均值缓存，训练推理逻辑完全一致
每张图片分布独立归一化，消除图片亮度、对比度差异

### 3. 优缺点 & 适用场景
✅ 优点：删除图片全局明暗信息，只保留纹理/结构风格
❌ 缺点：完全丢弃批量分布信息，分类任务效果极差
适用：图像风格迁移、GAN图像生成、滤镜渲染、纹理转换任务

In [ ]:
# ====================== InstanceNorm2d 风格迁移示例 ======================
# num_features：图像通道C
in_norm = nn.InstanceNorm2d(num_features=3).to(device)
# batch=3，3通道，64×64图片
style_imgs = torch.randn(3, 3, 64, 64).to(device)

# 训练模式
in_norm.train()
out_in_train = in_norm(style_imgs)
print("InstanceNorm2d训练输出shape:", out_in_train.shape)

# 单张风格图推理
single_style = torch.randn(1, 3, 64, 64).to(device)
in_norm.eval()
out_in_eval = in_norm(single_style)
print("InstanceNorm2d单图推理shape:", out_in_eval.shape)

# 关键特性：InstanceNorm 无可训练偏置beta
print("InstanceNorm可训练参数列表:", [name for name,_ in in_norm.named_parameters()])

## 四、三种归一化横向对比总表
| 归一化层 | 统计维度 | 是否依赖Batch | 有无全局缓存 | 可学习参数 | 核心适用场景 |
|----------|----------|---------------|--------------|------------|--------------|
| BatchNorm | 同通道跨全部Batch样本 | 强依赖 | 有running_mean/var | $\gamma,\beta$ | CNN图像分类、大batch |
| LayerNorm | 单样本全部特征维度 | 完全无关 | 无缓存 | $\gamma,\beta$ | Transformer、NLP、小batch |
| InstanceNorm | 单图单通道空间H/W像素 | 完全无关 | 无缓存 | 仅$\gamma$，无$\beta$ | 风格迁移、GAN图像生成 |

## 选型快速记忆
1. 图像分类、batch充足 → BatchNorm2d
2. NLP大模型、时序、小batch → LayerNorm
3. 风格转换、图像生成GAN → InstanceNorm2d